# Scam Classification Pipeline Reproducibility Notebook
This notebook demonstrates the end-to-end process of pulling raw data from DagsHub, compiling the Universal Stratified Dataset, training a sequence classification model (e.g., DistilBERT), and evaluating its accuracy on a 4,000-row held-out test set.

## 1. Setup Environment
First, we install the necessary libraries. We disable MLflow tracking for this run since this is just a local reproducibility test.

In [ ]:
!pip install transformers datasets pandas scikit-learn dagshub torch

import os
os.environ['DISABLE_MLFLOW_INTEGRATION'] = 'True'

## 2. Authenticate with DagsHub
We need to authenticate to pull the raw datasets from the DagsHub S3 bucket.

In [ ]:
import dagshub
from google.colab import userdata # if on Colab

# Ensure you have your DAGSHUB_TOKEN set
DAGSHUB_TOKEN = 'YOUR_TOKEN_HERE'
os.environ['DAGSHUB_USER'] = 'YOUR_USERNAME'
os.environ['DAGSHUB_TOKEN'] = DAGSHUB_TOKEN

repo_owner = 'tanu320' # Update if forked
repo_name = '2026SU_MS_DSP_422-DL_SEC61_Machine_Learning_Spam_detection'
repo_id = f'{repo_owner}/{repo_name}'
s3_client = dagshub.get_repo_bucket_client(repo_id)

## 3. Data Preparation & Curation
We download three distinct datasets (LLM Synthetic, Teeconnie, Legacy) and stratify them into a perfectly balanced 2,550-row training set and a 4,000-row test set. All rows are scrubbed of formatting leakage.

In [ ]:
import pandas as pd
import json, re, zipfile, glob, random

def clean_text(text):
    text = str(text)
    text = re.sub(r'(?i)(innocent|suspect):\s*', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

print("1. Loading LLM Data...")
s3_client.download_file(repo_name, 'data/raw_jsons/scam_call_hard_examples_250.json', 'hard.json')
s3_client.download_file(repo_name, 'data/raw_jsons/scam_call_transcripts_250_combined.json', 'comb.json')
synth_data = json.load(open('hard.json')) + json.load(open('comb.json'))
df_synth = pd.DataFrame(synth_data)
df_synth['text'] = df_synth['text'].apply(clean_text)

print("2. Loading Teeconnie Data...")
s3_client.download_file(repo_name, 'data/raw_teeconnie/teeconnie_dataset.zip', 'teeconnie.zip')
with zipfile.ZipFile('teeconnie.zip', 'r') as zipf: zipf.extractall("teeconnie_raw/")
txt_files = glob.glob("teeconnie_raw/**/*non*scam*.txt", recursive=True)
content = open(txt_files[0], "r", encoding="utf-8", errors="ignore").read()
entries = [e.strip() for e in content.split("\n\n") if e.strip()]
random.seed(42)
random.shuffle(entries)
df_tee = pd.DataFrame({"text": entries, "label": [0]*len(entries)})
df_tee['text'] = df_tee['text'].apply(clean_text)

print("3. Loading Legacy Data...")
s3_client.download_file(repo_name, 'data/legacy_composite/composite_train.csv', 'l_train.csv')
s3_client.download_file(repo_name, 'data/legacy_composite/composite_test.csv', 'l_test.csv')
df_leg = pd.concat([pd.read_csv('l_train.csv'), pd.read_csv('l_test.csv')]).dropna(subset=['text', 'label'])
df_leg['text'] = df_leg['text'].apply(clean_text)
leg_scam = df_leg[df_leg['label'] == 1].sample(frac=1, random_state=42)
leg_legit = df_leg[df_leg['label'] == 0].sample(frac=1, random_state=42)

print("4. Constructing Sets...")
train_df = pd.concat([
    df_synth[df_synth['label']==1], df_synth[df_synth['label']==0],
    df_tee.iloc[:350], leg_scam.iloc[:850], leg_legit.iloc[:850]
]).sample(frac=1, random_state=42).reset_index(drop=True)

test_df = pd.concat([
    leg_scam.iloc[850:2850], leg_legit.iloc[850:1850], df_tee.iloc[350:1350]
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Train Shape: {train_df.shape} | Test Shape: {test_df.shape}")

## 4. Model Training (DistilBERT)
We use DistilBERT here for fast training. We use the Hugging Face `Trainer` API without MLflow logging.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

# Convert Pandas to Huggingface Dataset
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_func(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

train_tok = train_ds.map(tokenize_func, batched=True)
test_tok = test_ds.map(tokenize_func, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

import evaluate
import numpy as np
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return clf_metrics.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    report_to="none" # Disables MLflow/WandB
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics
)

print("Starting Training...")
trainer.train()

## 5. Final Evaluation
Finally, we evaluate the trained model on our massive 4,000-row held-out test set to get the true accuracy metrics.

In [ ]:
print("Running Final Evaluation on Holdout Set...")
metrics = trainer.evaluate(eval_dataset=test_tok)
print("\n--- Final Metrics ---")
for k, v in metrics.items():
    print(f"{k}: {v}")